In [2]:
%load_ext line_profiler

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


In [3]:
import pandas as pd

In [4]:
embedding_dims = 32
batch_size = 32
n_transactions_context = 20  # the # of transactions to expect to be input in our model to generate a fraud prediction
dataset_training_frac = 0.7
random_state = 2020
train_test_split = 0.2
n_examples = 80000  # number of training/validation examples to generate
epochs = 32

#### Objective:

Train a classifier to predict if a given challenger transaction is from the same agency as `n_transactions_context` transactions provided.

#### Depends on:

In [ ]:
df_fname = "./data/05-dataframe-with-vectors.parquet
triplet_loss_model_fname = "./data/triplet_loss_model_4_agencies.pth"

#### Generates:

In [ ]:
fraud_detection_model_fname = "./data/fraud_detection_model_4_agencies.pth"

In [ ]:
X_test_output_fname = "./data/X_test_4_agencies.jbl"  # Testing data for later
y_test_output_fname = "./data/y_test_4_agencies.jbl"  # Testing data for later

--------------

In [5]:
df = pd.read_parquet(df_fname)  # this takes 4GB

In [6]:
# DEPARTMENT OF TRANSPORTATION|34500
# DEPARTMENT OF CORRECTIONS|13100
# DEPARTMENT OF TOURISM AND RECREATION|56600
# DEPARTMENT OF VETERANS AFFAIRS|65000

df = df[df.agency_number.isin([34500, 13100, 56600, 65000])]

In [7]:
df.head()

,cohort,agency_number,agency_name,holder_last_name,holder_first_initial,amount,vendor,mcc,parsed_amount,transaction_date,...,mcc_vector_758,mcc_vector_759,mcc_vector_760,mcc_vector_761,mcc_vector_762,mcc_vector_763,mcc_vector_764,mcc_vector_765,mcc_vector_766,mcc_vector_767
77998,201307,13100,DEPARTMENT OF CORRECTIONS,Briscoe,C,179.8,WW GRAINGER,INDUSTRIAL SUPPLIES NOT ELSEWHERE CLASSIFIED,5.191845,2013-07-25,...,-0.020916,-0.053247,0.054345,-0.017741,-0.044836,0.014481,0.016831,0.006052,-0.055828,-0.026703
77999,201307,13100,DEPARTMENT OF CORRECTIONS,Campbell,B,307.71,LOWES #02854,HOME SUPPLY WAREHOUSE STORES,5.729158,2013-07-24,...,-0.024244,0.011873,0.023957,-0.009558,0.002857,0.016128,-0.032343,-0.020793,-0.065192,-0.004960
78000,201307,13100,DEPARTMENT OF CORRECTIONS,Coats,R,885,YORK INTL OKC,INDUSTRIAL SUPPLIES NOT ELSEWHERE CLASSIFIED,6.785588,2013-07-25,...,-0.018412,0.009525,0.005979,-0.029377,-0.033671,-0.041418,0.006892,-0.003093,0.046676,-0.038271
78001,201307,13100,DEPARTMENT OF CORRECTIONS,Darrough,M,27.14,WANETTE TRACTOR & SUPPLY,"MISC. AUTOMOTIVE,AIRCRAFT,AND FARM EQUIPMENT D...",3.301009,2013-07-25,...,-0.018412,0.009525,0.005979,-0.029377,-0.033671,-0.041418,0.006892,-0.003093,0.046676,-0.038271
78002,201307,13100,DEPARTMENT OF CORRECTIONS,Darrough,M,102,INT CEDAR CREEK VET CLINI,VETERINARY SERVICES,4.624973,2013-07-25,...,-0.062032,-0.040161,0.060622,0.020844,-0.020162,-0.009240,0.019503,-0.005473,-0.098836,-0.019767


In [8]:
df = df.sample(frac=dataset_training_frac, random_state=random_state)

In [9]:
drop_cols = [
    "cohort",
    "agency_name",
    "holder_last_name",
    "holder_first_initial",
    "amount",
    "vendor",
    "mcc",
    "posted_date",
]  # keep 'transaction_date', 'agency_number', 'parsed_amount' and repr of mcc and vendor
label_col = "agency_number"

In [10]:
df.drop(drop_cols, axis=1, inplace=True)

In [11]:
input_dim = (
    len(df.columns) - 2
)  # 'transaction_date', 'agency_number' will not go into the model

In [12]:
import torch
import numpy as np
import random
from sklearn import model_selection
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
from torch.func import functional_call
import copy

In [13]:
class EmbeddingModel(nn.Module):
    def __init__(self, input_dim=1537, emb_dim=32):
        super(EmbeddingModel, self).__init__()

        self.fc = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.PReLU(),
            nn.Linear(input_dim, 1024),
            nn.PReLU(),
            nn.Linear(1024, 1024),
            nn.PReLU(),
            nn.Linear(1024, 128),
            nn.PReLU(),
            nn.Linear(128, 64),
            nn.PReLU(),
            nn.Linear(64, 64),
            nn.PReLU(),
            nn.Linear(64, emb_dim),
        )

    def forward(self, x):
        x = self.fc(x)
        return x

In [14]:
torch.manual_seed(random_state)
np.random.seed(random_state)
random.seed(random_state)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    torch.cuda.get_device_name()

In [15]:
emb_model = EmbeddingModel(input_dim, embedding_dims)

In [16]:
emb_model.load_state_dict(torch.load(triplet_loss_model_fname))

# emb_model = torch.jit.script(emb_model).to(device)

<All keys matched successfully>

In [17]:
emb_model.eval()

EmbeddingModel(
  (fc): Sequential(
    (0): Linear(in_features=1537, out_features=1537, bias=True)
    (1): PReLU(num_parameters=1)
    (2): Linear(in_features=1537, out_features=1024, bias=True)
    (3): PReLU(num_parameters=1)
    (4): Linear(in_features=1024, out_features=1024, bias=True)
    (5): PReLU(num_parameters=1)
    (6): Linear(in_features=1024, out_features=128, bias=True)
    (7): PReLU(num_parameters=1)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): PReLU(num_parameters=1)
    (10): Linear(in_features=64, out_features=64, bias=True)
    (11): PReLU(num_parameters=1)
    (12): Linear(in_features=64, out_features=32, bias=True)
  )
)

In [18]:
with torch.no_grad():
    sample = df.sample(1).drop(["transaction_date", "agency_number"], axis=1).values
    sample_embedding = emb_model(torch.from_numpy(sample).cpu()).cpu().numpy()

sample_embedding

array([[ 0.3817034 ,  0.21766295,  0.23239535,  0.15711193,  0.23869261,
        -0.22828142,  0.59422565, -0.27739024,  0.5426107 ,  0.37793866,
         0.13959832,  0.5837817 ,  0.2379188 ,  0.3217694 , -0.59396315,
        -0.31058773,  0.22172114, -0.06382404,  0.18322298,  0.15122727,
         0.2777243 , -0.30660826, -0.20724763, -0.27246144,  0.26987317,
         0.27777562,  0.02516324,  0.4145733 , -0.02961469, -0.17387877,
        -0.713159  ,  0.5643669 ]], dtype=float32)

In [19]:
for param in emb_model.parameters():
    param.requires_grad = False

In [20]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight, gain=nn.init.calculate_gain("relu"))
        m.bias.data.fill_(0.01)


class Classifier(nn.Module):
    def __init__(self, embedding_model, n_transactions_context, embedding_dims):
        super(Classifier, self).__init__()

        self.n_transactions_context = n_transactions_context
        self.embedding_dims = embedding_dims

        emb_models = []

        for i in range(n_transactions_context + 1):
            emb_models.append(copy.deepcopy(embedding_model))

        self.emb_models = nn.ModuleList(emb_models)

        self.fc = nn.Sequential(
            nn.Linear((n_transactions_context + 1) * embedding_dims, 512),
            nn.PReLU(),
            nn.Linear(512, 256),
            nn.PReLU(),
            nn.Linear(256, 128),
            nn.PReLU(),
            nn.Linear(128, 32),
            nn.PReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

        self.fc.apply(init_weights)

    def forward(self, x):
        # print(x.shape)

        embs = [model(x[:, i]) for i, model in enumerate(self.emb_models)]

        # print("len(embs)", len(embs))

        # print("embs[0].shape", embs[0].shape)

        cat = torch.cat(embs, dim=1)

        # print("cat.shape", cat.shape)

        flat = cat.flatten(start_dim=1)

        # print("flat.shape", flat.shape)

        x = self.fc(flat)

        return x

In [21]:
cl_model = Classifier(emb_model, n_transactions_context, embedding_dims)

In [22]:
cl_model = torch.jit.script(cl_model).to(device)

In [23]:
with torch.no_grad():
    sample = np.array(
        [
            df.sample(21).drop(["transaction_date", "agency_number"], axis=1).values,
            df.sample(21).drop(["transaction_date", "agency_number"], axis=1).values,
        ]
    )
    # print(sample.shape)
    sample_prediction = cl_model(torch.from_numpy(sample).to(device)).cpu().numpy()

sample_prediction

array([[0.4266611],
       [0.6845571]], dtype=float32)

In [24]:
df.head()

,agency_number,parsed_amount,transaction_date,vendor_vector_0,vendor_vector_1,vendor_vector_2,vendor_vector_3,vendor_vector_4,vendor_vector_5,vendor_vector_6,...,mcc_vector_758,mcc_vector_759,mcc_vector_760,mcc_vector_761,mcc_vector_762,mcc_vector_763,mcc_vector_764,mcc_vector_765,mcc_vector_766,mcc_vector_767
134039,56600,6.126695,2013-08-09,0.012147,0.021508,-0.007825,-0.007432,-0.002976,0.015324,0.052413,...,-0.004007,0.013374,0.047400,-0.004896,-0.032428,-0.033051,0.014269,-0.003037,0.002706,0.019527
90263,13100,4.332968,2014-01-08,0.012147,0.021508,-0.007825,-0.007432,-0.002976,0.015324,0.052413,...,0.003073,-0.002374,0.028515,0.016387,-0.000060,0.044012,-0.040820,-0.041668,-0.027988,-0.018353
397298,34500,4.440296,2014-05-01,-0.016400,0.056949,-0.029707,0.003588,-0.015921,-0.007672,0.051330,...,-0.062107,-0.056114,0.043701,-0.010523,-0.035250,-0.020894,-0.037667,-0.030074,0.033692,0.004435
150581,65000,6.199494,2013-11-13,0.012147,0.021508,-0.007825,-0.007432,-0.002976,0.015324,0.052413,...,-0.025653,-0.027870,0.019045,-0.032145,-0.051140,0.009164,0.020847,0.035085,-0.029779,-0.018159
85353,13100,6.260231,2013-10-01,0.012147,0.021508,-0.007825,-0.007432,-0.002976,0.015324,0.052413,...,0.003073,-0.002374,0.028515,0.016387,-0.000060,0.044012,-0.040820,-0.041668,-0.027988,-0.018353


In [25]:
df = df.set_index(["agency_number", "transaction_date"], drop=False).sort_index()

In [26]:
df.head()

agency_number  parsed_amount transaction_date  \
agency_number transaction_date                                                  
13100         2013-06-24                13100       6.962811       2013-06-24   
              2013-06-26                13100       3.087399       2013-06-26   
              2013-06-26                13100       6.192342       2013-06-26   
              2013-06-26                13100       4.828314       2013-06-26   
              2013-06-27                13100       4.248353       2013-06-27   

                                vendor_vector_0  vendor_vector_1  \
agency_number transaction_date                                     
13100         2013-06-24               0.008680        -0.008850   
              2013-06-26               0.008680        -0.008850   
              2013-06-26               0.012147         0.021508   
              2013-06-26               0.012147         0.021508   
              2013-06-27               0.012147         0.021508   

                                vendor_vector_2  vendor_vector_3  \
agency_number transaction_date                                     
13100         2013-06-24              -0.005182        -0.019806   
              2013-06-26              -0.005182        -0.019806   
              2013-06-26              -0.007825        -0.007432   
              2013-06-26              -0.007825        -0.007432   
              2013-06-27              -0.007825        -0.007432   

                                vendor_vector_4  vendor_vector_5  \
agency_number transaction_date                                     
13100         2013-06-24               0.006723         0.024422   
              2013-06-26               0.006723         0.024422   
              2013-06-26              -0.002976         0.015324   
              2013-06-26              -0.002976         0.015324   
              2013-06-27              -0.002976         0.015324   

                                vendor_vector_6  ...  mcc_vector_758  \
agency_number transaction_date                   ...                   
13100         2013-06-24               0.071407  ...       -0.020916   
              2013-06-26               0.071407  ...        0.002987   
              2013-06-26               0.052413  ...       -0.057629   
              2013-06-26               0.052413  ...       -0.020916   
              2013-06-27               0.052413  ...       -0.001497   

                                mcc_vector_759  mcc_vector_760  \
agency_number transaction_date                                   
13100         2013-06-24             -0.053247        0.054345   
              2013-06-26             -0.002214        0.009767   
              2013-06-26              0.027079        0.027732   
              2013-06-26             -0.053247        0.054345   
              2013-06-27             -0.017025        0.025489   

                                mcc_vector_761  mcc_vector_762  \
agency_number transaction_date                                   
13100         2013-06-24             -0.017741       -0.044836   
              2013-06-26              0.008471       -0.013761   
              2013-06-26             -0.015218       -0.034617   
              2013-06-26             -0.017741       -0.044836   
              2013-06-27              0.002128       -0.002501   

                                mcc_vector_763  mcc_vector_764  \
agency_number transaction_date                                   
13100         2013-06-24              0.014481        0.016831   
              2013-06-26              0.033304       -0.034486   
              2013-06-26             -0.040562        0.026749   
              2013-06-26              0.014481        0.016831   
              2013-06-27             -0.015514        0.026597   

                                mcc_vector_765  mcc_vector_766  mcc_vector_767  
agency_number transaction_date                            

In [27]:
# df.loc[1000][:'2013-06-26']

In [28]:
agencies = set(df.agency_number.unique())

In [29]:
X = np.zeros(
    (n_examples, n_transactions_context + 1, len(df.columns) - 2), dtype="float"
)
y = np.zeros((n_examples,), dtype="float")

In [30]:
y.shape

(80000,)

In [31]:
X.shape

(80000, 21, 1537)

Generates `n_examples` examples by:

1. Getting a Transaction from the Dataset 
2. Finingd 20 previous transactions from the same agency
3. Creingte a positive example from 1 and 2
4. Saingple a Transaction from another agency
5. Cingeate a negative example from 2 and 4.

In [32]:
for i in tqdm(range(0, n_examples, 2)):
    not_enough_transactions = True

    while not_enough_transactions:
        transaction = df.sample(1)

        try:
            earlier_transactions = df.loc[transaction.agency_number.values[0]][
                : transaction.transaction_date.values[0]
            ].sample(n_transactions_context)
            not_enough_transactions = False
        except ValueError:
            # print("sample new transaction")
            continue  # sample new transaction

    # positive example
    X[i, :n_transactions_context, :] = earlier_transactions.drop(
        ["transaction_date", "agency_number"], axis=1
    ).values
    X[i, n_transactions_context, :] = transaction.drop(
        ["transaction_date", "agency_number"], axis=1
    ).values
    y[i] = 1  # same agency

    negative_agency = np.random.choice(
        list(agencies.difference(transaction.agency_number.values))
    )

    # negative example
    negative_example = df.loc[negative_agency].sample(1)

    X[i + 1, :n_transactions_context, :] = earlier_transactions.drop(
        ["transaction_date", "agency_number"], axis=1
    ).values
    X[i + 1, n_transactions_context, :] = negative_example.drop(
        ["transaction_date", "agency_number"], axis=1
    ).values
    y[i + 1] = 0  # not the same agency

  0%|          | 0/40000 [00:00<?, ?it/s]

In [33]:
del df

In [34]:
class TransactionsDataset(Dataset):
    def __init__(self, X, y, train=True):
        self.X = X
        self.y = y
        self.train = train

    def __len__(self):
        return len(self.X)

    def __getitem__(self, item):
        return self.X[item].astype("float32"), self.y[item].astype("float32")

In [35]:
# X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=train_test_split, random_state=random_state, shuffle=False) # HUGE MEMORY COST

X_test = X[: int(len(X) * train_test_split)]
y_test = y[: int(len(X) * train_test_split)]
X_train = X[int(len(X) * train_test_split) :]
y_train = y[int(len(X) * train_test_split) :]

In [36]:
len(X_train), len(X_test)

(64000, 16000)

In [37]:
del X
del y

In [38]:
train_ds = TransactionsDataset(X_train, y_train, train=True)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)

In [39]:
# test_ds = TransactionsDataset(X_test, y_train, train=False)
# test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=True, num_workers=4)

In [40]:
import joblib

with open(X_test_output_fname, "wb") as f:
    joblib.dump(
        X_test,
        f,
    )

with open(y_test_output_fname, "wb") as f:
    joblib.dump(
        y_test,
        f,
    )

In [ ]:
optimizer = optim.Adam(cl_model.parameters(), lr=0.001)
criterion = nn.BCELoss()

In [ ]:
cl_model.train()
for epoch in tqdm(range(epochs), desc="Epochs"):
    running_loss = []
    for step, (X_batch, y_batch) in enumerate(
        tqdm(train_loader, desc="Training", leave=False)
    ):
        X_tensor = X_batch.to(device)
        y_tensor = y_batch.reshape(-1, 1).to(device)

        # print("X_tensor", X_tensor)
        # print("y_tensor", y_tensor)

        optimizer.zero_grad()

        model_out = cl_model(X_tensor)

        # print("model_out", y_tensor)

        loss = criterion(model_out, y_tensor)

        # print("loss", loss)

        loss.backward()
        optimizer.step()

        running_loss.append(loss.cpu().detach().numpy())
    print(
        "Epoch: {}/{} - Loss: {:.4f}".format(epoch + 1, epochs, np.mean(running_loss))
    )

In [ ]:
print(X_tensor.shape)

In [ ]:
torch.save(cl_model.state_dict(), fraud_detection_model_fname)